In [104]:
!pip -q install -U transformers datasets evaluate accelerate soundfile librosa scikit-learn pandas matplotlib seaborn

In [105]:
import os
import json
import math
import random
import warnings
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import librosa

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoFeatureExtractor,
    WavLMModel,
    get_linear_schedule_with_warmup,
)

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

In [106]:
# =========================
# CONFIG
# =========================
SEED = 42
TARGET_SR = 16000
TARGET_SECONDS = 3.0
TARGET_NUM_SAMPLES = int(TARGET_SR * TARGET_SECONDS)

# Dataset location
DATA_ROOT = "/content/drive/MyDrive/data/wavlm_dataset"
METADATA_CSV = "/content/drive/MyDrive/data/wavlm_dataset/metadata.csv"

# Save outputs to Drive
OUTPUT_DIR = "/content/drive/MyDrive/data/wavlm_runs"
RUN_NAME = "wavlm_multitask_region_gender"
RUN_DIR = os.path.join(OUTPUT_DIR, RUN_NAME)

MODEL_NAME = "microsoft/wavlm-base-plus"

BATCH_SIZE = 8
GRAD_ACCUM_STEPS = 2
NUM_EPOCHS = 8
LR = 1e-5
WEIGHT_DECAY = 1e-4
WARMUP_RATIO = 0.1
MAX_GRAD_NORM = 1.0

FREEZE_FEATURE_EXTRACTOR = True
FREEZE_ENCODER_N_LAYERS = 6
USE_CLASS_WEIGHTS = True

NUM_WORKERS = 2
PIN_MEMORY = True

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

REGION_ID2LABEL = {0: "North", 1: "Central", 2: "South"}
REGION_LABEL2ID = {v: k for k, v in REGION_ID2LABEL.items()}

GENDER_ID2LABEL = {0: "Male", 1: "Female"}
GENDER_LABEL2ID = {v: k for k, v in GENDER_ID2LABEL.items()}

os.makedirs(RUN_DIR, exist_ok=True)

print("DEVICE:", DEVICE)
print("RUN_DIR:", RUN_DIR)
print("DATA_ROOT:", DATA_ROOT)
print("METADATA_CSV:", METADATA_CSV)

DEVICE: cuda
RUN_DIR: /content/drive/MyDrive/data/wavlm_runs/wavlm_multitask_region_gender
DATA_ROOT: /content/drive/MyDrive/data/wavlm_dataset
METADATA_CSV: /content/drive/MyDrive/data/wavlm_dataset/metadata.csv


In [107]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

In [108]:
def pad_or_trim_audio(audio: np.ndarray, target_len: int) -> np.ndarray:
    if len(audio) < target_len:
        pad_width = target_len - len(audio)
        audio = np.pad(audio, (0, pad_width), mode="constant")
    else:
        audio = audio[:target_len]
    return audio

def rms(x: np.ndarray) -> float:
    return float(np.sqrt(np.mean(np.square(x), dtype=np.float64) + 1e-12))

def match_loudness(audio: np.ndarray, target_rms: float = 0.015823712572455406) -> np.ndarray:
    current_rms = rms(audio)
    if current_rms < 1e-10:
        return audio
    scale = target_rms / current_rms
    return audio * scale

def normalize_audio(audio: np.ndarray) -> np.ndarray:
    max_abs = np.max(np.abs(audio)) + 1e-12
    return (audio / max_abs).astype(np.float32)

def load_audio_safely(path: str, target_sr: int = 16000, target_num_samples: int = 48000):
    audio, sr = sf.read(path)

    if audio.ndim == 2:
        audio = np.mean(audio, axis=1)

    audio = audio.astype(np.float32)

    if sr != target_sr:
        audio = librosa.resample(audio, orig_sr=sr, target_sr=target_sr)
        sr = target_sr

    audio = pad_or_trim_audio(audio, target_num_samples)
    audio = match_loudness(audio)
    audio = normalize_audio(audio)
    audio = pad_or_trim_audio(audio, target_num_samples).astype(np.float32)

    return audio, sr

In [109]:
print("region_id unique:", sorted(df["region_id"].dropna().unique().tolist()))
print("gender_id unique:", sorted(df["gender_id"].dropna().unique().tolist()))
print(df[["region", "region_id", "gender_id"]].drop_duplicates().sort_values(["region_id", "gender_id"]).head(20))

region_id unique: [0, 1, 2]
gender_id unique: [0, 1]
       region  region_id  gender_id
25      North          0          0
0       North          0          1
2857  Central          1          0
2858  Central          1          1
5644    South          2          0
5634    South          2          1


In [110]:
df = pd.read_csv(METADATA_CSV)

required_cols = ["path", "split", "region", "region_id", "gender_id"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns in metadata.csv: {missing}")

df["split"] = df["split"].astype(str).str.lower().str.strip()

valid_splits = {"train", "val", "test"}
bad_splits = sorted(set(df["split"].unique()) - valid_splits)
if bad_splits:
    raise ValueError(f"Unexpected split values: {bad_splits}")

df["region_id"] = df["region_id"].astype(int)
df["gender_id"] = df["gender_id"].astype(int)

print("Original region_id unique:", sorted(df["region_id"].dropna().unique().tolist()))
print("Original gender_id unique:", sorted(df["gender_id"].dropna().unique().tolist()))

region_id_map = {1: 0, 2: 1, 3: 2}
region_name_to_expected_id = {
    "North": 0,
    "Central": 1,
    "South": 2,
}

region_unique = set(df["region_id"].unique().tolist())
if region_unique == {1, 2, 3}:
    df["region_id"] = df["region_id"].map(region_id_map)
    print("Mapped region_id from {1,2,3} to {0,1,2}")
elif region_unique == {0, 1, 2}:
    print("region_id already in {0,1,2}")
else:
    raise ValueError(
        f"Unsupported region_id values: {sorted(region_unique)}. "
        f"Expected either [1,2,3] or [0,1,2]."
    )

gender_unique = set(df["gender_id"].unique().tolist())
if gender_unique != {0, 1}:
    raise ValueError(
        f"Unsupported gender_id values: {sorted(gender_unique)}. "
        f"Expected [0,1]."
    )

region_check = (
    df[["region", "region_id"]]
    .drop_duplicates()
    .sort_values(["region_id", "region"])
    .reset_index(drop=True)
)

print("\nRegion text/id pairs after normalization:")
print(region_check)

for _, row in region_check.iterrows():
    region_name = str(row["region"]).strip()
    region_id = int(row["region_id"])
    if region_name in region_name_to_expected_id:
        expected_id = region_name_to_expected_id[region_name]
        if region_id != expected_id:
            raise ValueError(
                f"Region label mismatch: region='{region_name}' has region_id={region_id}, "
                f"expected {expected_id}."
            )

assert set(df["region_id"].unique()).issubset({0, 1, 2}), "region_id must be in {0,1,2}"
assert set(df["gender_id"].unique()).issubset({0, 1}), "gender_id must be in {0,1}"

def resolve_path(p):
    p = str(p).strip()

    if os.path.exists(p):
        return p

    alt = os.path.join(DATA_ROOT, p)
    if os.path.exists(alt):
        return alt

    if p.startswith("wavlm_dataset/"):
        alt2 = os.path.join("/content/drive/MyDrive/data", p)
        if os.path.exists(alt2):
            return alt2

    return p

df["path"] = df["path"].apply(resolve_path)

missing_files = df[~df["path"].apply(os.path.exists)]
print("\nTotal rows:", len(df))
print("Missing files:", len(missing_files))

if len(missing_files) > 0:
    display(missing_files.head(10))
    raise FileNotFoundError("Some audio files in metadata.csv do not exist.")

print("\nSplit counts:")
print(df["split"].value_counts())

print("\nRegion distribution by split:")
print(pd.crosstab(df["split"], df["region"]))

print("\nGender distribution by split:")
print(pd.crosstab(df["split"], df["gender_id"]))

print("\nNormalized region_id unique:", sorted(df["region_id"].unique().tolist()))
print("Normalized gender_id unique:", sorted(df["gender_id"].unique().tolist()))

Original region_id unique: [1, 2, 3]
Original gender_id unique: [0, 1]
Mapped region_id from {1,2,3} to {0,1,2}

Region text/id pairs after normalization:
    region  region_id
0    North          0
1  Central          1
2    South          2

Total rows: 16156
Missing files: 0

Split counts:
split
train    10064
val       3140
test      2952
Name: count, dtype: int64

Region distribution by split:
region  Central  North  South
split                        
test        848   1252    852
train      3602   3362   3100
val        1104   1100    936

Gender distribution by split:
gender_id     0     1
split                
test       1967   985
train      6644  3420
val        2111  1029

Normalized region_id unique: [0, 1, 2]
Normalized gender_id unique: [0, 1]


In [111]:
sample_n = min(20, len(df))
sample_df = df.sample(sample_n, random_state=SEED)

problems = []

for _, row in sample_df.iterrows():
    try:
        audio, sr = load_audio_safely(row["path"], TARGET_SR, TARGET_NUM_SAMPLES)
        if sr != TARGET_SR:
            problems.append((row["path"], f"sr={sr}"))
        if len(audio) != TARGET_NUM_SAMPLES:
            problems.append((row["path"], f"len={len(audio)}"))
    except Exception as e:
        problems.append((row["path"], str(e)))

print("Sanity check sample size:", sample_n)
print("Problems found:", len(problems))
if problems:
    for p in problems[:10]:
        print(p)

Sanity check sample size: 20
Problems found: 0


In [112]:
train_df = df[df["split"] == "train"].reset_index(drop=True).copy()
val_df   = df[df["split"] == "val"].reset_index(drop=True).copy()
test_df  = df[df["split"] == "test"].reset_index(drop=True).copy()

print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))

Train: 10064
Val: 3140
Test: 2952


In [113]:
feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_NAME)
print(feature_extractor)

Wav2Vec2FeatureExtractor {
  "do_normalize": false,
  "feature_extractor_type": "Wav2Vec2FeatureExtractor",
  "feature_size": 1,
  "padding_side": "right",
  "padding_value": 0.0,
  "return_attention_mask": true,
  "sampling_rate": 16000
}



In [114]:
class VietnameseSpeechDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, feature_extractor, target_sr=16000, target_num_samples=48000):
        self.df = dataframe.reset_index(drop=True).copy()
        self.feature_extractor = feature_extractor
        self.target_sr = target_sr
        self.target_num_samples = target_num_samples

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        audio, sr = load_audio_safely(
            row["path"],
            target_sr=self.target_sr,
            target_num_samples=self.target_num_samples
        )

        inputs = self.feature_extractor(
            audio,
            sampling_rate=sr,
            return_attention_mask=True,
            return_tensors="pt"
        )

        item = {
            "input_values": inputs["input_values"].squeeze(0),
            "attention_mask": inputs["attention_mask"].squeeze(0),
            "region_labels": torch.tensor(int(row["region_id"]), dtype=torch.long),
            "gender_labels": torch.tensor(int(row["gender_id"]), dtype=torch.long),
            "path": row["path"],
            "split": row["split"],
            "region_id_true": int(row["region_id"]),
            "gender_id_true": int(row["gender_id"]),
        }
        return item

In [115]:
@dataclass
class AudioCollator:
    feature_extractor: object

    def __call__(self, features):
        input_values = [f["input_values"] for f in features]
        attention_mask = [f["attention_mask"] for f in features]

        batch = self.feature_extractor.pad(
            {
                "input_values": input_values,
                "attention_mask": attention_mask,
            },
            padding=True,
            return_tensors="pt"
        )

        batch["region_labels"] = torch.stack([f["region_labels"] for f in features])
        batch["gender_labels"] = torch.stack([f["gender_labels"] for f in features])

        batch["path"] = [f["path"] for f in features]
        batch["split"] = [f["split"] for f in features]
        batch["region_id_true"] = [f["region_id_true"] for f in features]
        batch["gender_id_true"] = [f["gender_id_true"] for f in features]

        return batch

In [116]:
train_dataset = VietnameseSpeechDataset(train_df, feature_extractor, TARGET_SR, TARGET_NUM_SAMPLES)
val_dataset   = VietnameseSpeechDataset(val_df, feature_extractor, TARGET_SR, TARGET_NUM_SAMPLES)
test_dataset  = VietnameseSpeechDataset(test_df, feature_extractor, TARGET_SR, TARGET_NUM_SAMPLES)

collator = AudioCollator(feature_extractor)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    collate_fn=collator
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    collate_fn=collator
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    collate_fn=collator
)

print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))
print("Test batches:", len(test_loader))

Train batches: 1258
Val batches: 393
Test batches: 369


In [117]:
def make_class_weights(labels, num_classes):
    counts = np.bincount(labels, minlength=num_classes).astype(np.float32)
    weights = counts.sum() / (num_classes * np.maximum(counts, 1.0))
    return torch.tensor(weights, dtype=torch.float32)

region_class_weights = make_class_weights(train_df["region_id"].values, 3)
gender_class_weights = make_class_weights(train_df["gender_id"].values, 2)

print("Region class weights:", region_class_weights)
print("Gender class weights:", gender_class_weights)

Region class weights: tensor([0.9978, 0.9313, 1.0822])
Gender class weights: tensor([0.7574, 1.4713])


In [118]:
class WavLMMultiTaskClassifier(nn.Module):
    def __init__(
        self,
        model_name: str,
        num_region_labels: int = 3,
        num_gender_labels: int = 2,
        dropout: float = 0.2,
        freeze_feature_extractor: bool = True,
        freeze_encoder_n_layers: int = 0,
        region_class_weights: torch.Tensor = None,
        gender_class_weights: torch.Tensor = None,
    ):
        super().__init__()

        self.wavlm = WavLMModel.from_pretrained(model_name)
        hidden_size = self.wavlm.config.hidden_size

        self.dropout = nn.Dropout(dropout)

        self.region_head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, num_region_labels),
        )

        self.gender_head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, num_gender_labels),
        )

        self.region_loss_fn = nn.CrossEntropyLoss(
            weight=region_class_weights if USE_CLASS_WEIGHTS else None
        )
        self.gender_loss_fn = nn.CrossEntropyLoss(
            weight=gender_class_weights if USE_CLASS_WEIGHTS else None
        )

        if freeze_feature_extractor:
            try:
                self.wavlm.feature_extractor._freeze_parameters()
            except Exception:
                pass

        if freeze_encoder_n_layers > 0:
            for layer in self.wavlm.encoder.layers[:freeze_encoder_n_layers]:
                for param in layer.parameters():
                    param.requires_grad = False

    def masked_mean_pool(self, hidden_states, attention_mask=None):
        if attention_mask is None:
            return hidden_states.mean(dim=1)

        if hasattr(self.wavlm, "_get_feature_vector_attention_mask"):
            feat_attention_mask = self.wavlm._get_feature_vector_attention_mask(
                hidden_states.shape[1],
                attention_mask
            )
        else:
            feat_attention_mask = F.interpolate(
                attention_mask.unsqueeze(1).float(),
                size=hidden_states.shape[1],
                mode="nearest"
            ).squeeze(1).long()

        mask = feat_attention_mask.unsqueeze(-1).to(hidden_states.dtype)
        summed = torch.sum(hidden_states * mask, dim=1)
        counts = torch.clamp(mask.sum(dim=1), min=1e-9)
        return summed / counts

    def forward(self, input_values, attention_mask, region_labels=None, gender_labels=None):
        outputs = self.wavlm(
            input_values=input_values,
            attention_mask=attention_mask
        )

        hidden_states = outputs.last_hidden_state
        pooled = self.masked_mean_pool(hidden_states, attention_mask)
        pooled = self.dropout(pooled)

        region_logits = self.region_head(pooled)
        gender_logits = self.gender_head(pooled)

        loss = None
        region_loss = None
        gender_loss = None

        if region_labels is not None and gender_labels is not None:
            region_loss = self.region_loss_fn(region_logits, region_labels)
            gender_loss = self.gender_loss_fn(gender_logits, gender_labels)
            loss = region_loss + gender_loss

        return {
            "loss": loss,
            "region_loss": region_loss,
            "gender_loss": gender_loss,
            "region_logits": region_logits,
            "gender_logits": gender_logits,
        }

In [119]:
model = WavLMMultiTaskClassifier(
    model_name=MODEL_NAME,
    num_region_labels=3,
    num_gender_labels=2,
    dropout=0.2,
    freeze_feature_extractor=FREEZE_FEATURE_EXTRACTOR,
    freeze_encoder_n_layers=FREEZE_ENCODER_N_LAYERS,
    region_class_weights=region_class_weights.to(DEVICE),
    gender_class_weights=gender_class_weights.to(DEVICE),
).to(DEVICE)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())

print(f"Trainable params: {trainable:,}")
print(f"Total params:     {total:,}")

Loading weights:   0%|          | 0/248 [00:00<?, ?it/s]

Trainable params: 48,832,253
Total params:     95,566,965


In [120]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

total_train_steps = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS) * NUM_EPOCHS
warmup_steps = int(total_train_steps * WARMUP_RATIO)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_train_steps
)

print("Total train steps:", total_train_steps)
print("Warmup steps:", warmup_steps)

Total train steps: 5032
Warmup steps: 503


In [121]:
def compute_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted"),
    }

def save_confusion_matrix(cm, class_names, title, out_path):
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names)
    plt.title(title)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close()

def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

In [122]:
@torch.no_grad()
def run_inference(model, data_loader, device):
    model.eval()

    all_region_true = []
    all_gender_true = []
    all_region_pred = []
    all_gender_pred = []
    all_region_probs = []
    all_gender_probs = []
    all_rows = []

    total_loss = 0.0
    steps = 0

    for batch in data_loader:
        input_values = batch["input_values"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        region_labels = batch["region_labels"].to(device)
        gender_labels = batch["gender_labels"].to(device)

        outputs = model(
            input_values=input_values,
            attention_mask=attention_mask,
            region_labels=region_labels,
            gender_labels=gender_labels,
        )

        if outputs["loss"] is not None:
            total_loss += outputs["loss"].item()
            steps += 1

        region_probs = torch.softmax(outputs["region_logits"], dim=-1).cpu().numpy()
        gender_probs = torch.softmax(outputs["gender_logits"], dim=-1).cpu().numpy()

        region_pred = np.argmax(region_probs, axis=1)
        gender_pred = np.argmax(gender_probs, axis=1)

        region_true = batch["region_labels"].cpu().numpy()
        gender_true = batch["gender_labels"].cpu().numpy()

        all_region_true.extend(region_true.tolist())
        all_gender_true.extend(gender_true.tolist())
        all_region_pred.extend(region_pred.tolist())
        all_gender_pred.extend(gender_pred.tolist())
        all_region_probs.extend(region_probs.tolist())
        all_gender_probs.extend(gender_probs.tolist())

        for i in range(len(batch["path"])):
            all_rows.append({
                "path": batch["path"][i],
                "split": batch["split"][i],
                "region_id_true": int(batch["region_id_true"][i]),
                "gender_id_true": int(batch["gender_id_true"][i]),
                "region_id_pred": int(region_pred[i]),
                "gender_id_pred": int(gender_pred[i]),
                "region_probs": [float(x) for x in region_probs[i]],
                "gender_probs": [float(x) for x in gender_probs[i]],
            })

    region_metrics = compute_metrics(all_region_true, all_region_pred)
    gender_metrics = compute_metrics(all_gender_true, all_gender_pred)

    avg_loss = total_loss / max(steps, 1)

    return {
        "loss": avg_loss,
        "region_true": all_region_true,
        "region_pred": all_region_pred,
        "gender_true": all_gender_true,
        "gender_pred": all_gender_pred,
        "region_probs": all_region_probs,
        "gender_probs": all_gender_probs,
        "rows": all_rows,
        "region_metrics": region_metrics,
        "gender_metrics": gender_metrics,
    }

In [95]:
def train_one_epoch(model, data_loader, optimizer, scheduler, device, grad_accum_steps=1, max_grad_norm=1.0):
    model.train()
    total_loss = 0.0

    optimizer.zero_grad()

    for step, batch in enumerate(data_loader):
        input_values = batch["input_values"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        region_labels = batch["region_labels"].to(device)
        gender_labels = batch["gender_labels"].to(device)

        outputs = model(
            input_values=input_values,
            attention_mask=attention_mask,
            region_labels=region_labels,
            gender_labels=gender_labels,
        )

        loss = outputs["loss"] / grad_accum_steps
        loss.backward()

        if (step + 1) % grad_accum_steps == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        total_loss += outputs["loss"].item()

    return total_loss / max(len(data_loader), 1)


best_score = -1.0
best_model_path = os.path.join(RUN_DIR, "best_model.pt")
history = []

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss = train_one_epoch(
        model, train_loader, optimizer, scheduler, DEVICE,
        grad_accum_steps=GRAD_ACCUM_STEPS,
        max_grad_norm=MAX_GRAD_NORM
    )

    val_out = run_inference(model, val_loader, DEVICE)

    joint_score = (
        val_out["region_metrics"]["macro_f1"] +
        val_out["gender_metrics"]["macro_f1"]
    ) / 2.0

    epoch_log = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_out["loss"],
        "val_region_accuracy": val_out["region_metrics"]["accuracy"],
        "val_region_macro_f1": val_out["region_metrics"]["macro_f1"],
        "val_region_weighted_f1": val_out["region_metrics"]["weighted_f1"],
        "val_gender_accuracy": val_out["gender_metrics"]["accuracy"],
        "val_gender_macro_f1": val_out["gender_metrics"]["macro_f1"],
        "val_gender_weighted_f1": val_out["gender_metrics"]["weighted_f1"],
        "joint_score": joint_score,
    }
    history.append(epoch_log)

    print(
        f"Epoch {epoch}/{NUM_EPOCHS} | "
        f"train_loss={train_loss:.4f} | val_loss={val_out['loss']:.4f} | "
        f"region_acc={val_out['region_metrics']['accuracy']:.4f} | "
        f"region_macro_f1={val_out['region_metrics']['macro_f1']:.4f} | "
        f"gender_acc={val_out['gender_metrics']['accuracy']:.4f} | "
        f"gender_macro_f1={val_out['gender_metrics']['macro_f1']:.4f}"
    )

    if joint_score > best_score:
        best_score = joint_score
        torch.save(model.state_dict(), best_model_path)
        print(f"  Saved new best model to: {best_model_path}")

history_df = pd.DataFrame(history)
history_df.to_csv(os.path.join(RUN_DIR, "training_history.csv"), index=False)
history_df.tail()

Epoch 1/8 | train_loss=1.6168 | val_loss=1.2603 | region_acc=0.4497 | region_macro_f1=0.4426 | gender_acc=0.9373 | gender_macro_f1=0.9312
Epoch 1/8 | train_loss=1.6168 | val_loss=1.2603 | region_acc=0.4497 | region_macro_f1=0.4426 | gender_acc=0.9373 | gender_macro_f1=0.9312
  Saved new best model to: /content/drive/MyDrive/data/wavlm_runs/wavlm_multitask_region_gender/best_model.pt
  Saved new best model to: /content/drive/MyDrive/data/wavlm_runs/wavlm_multitask_region_gender/best_model.pt
Epoch 2/8 | train_loss=1.1855 | val_loss=1.3485 | region_acc=0.4519 | region_macro_f1=0.4455 | gender_acc=0.8943 | gender_macro_f1=0.8875
Epoch 2/8 | train_loss=1.1855 | val_loss=1.3485 | region_acc=0.4519 | region_macro_f1=0.4455 | gender_acc=0.8943 | gender_macro_f1=0.8875
Epoch 3/8 | train_loss=1.0049 | val_loss=1.1920 | region_acc=0.5787 | region_macro_f1=0.5789 | gender_acc=0.8965 | gender_macro_f1=0.8900
Epoch 3/8 | train_loss=1.0049 | val_loss=1.1920 | region_acc=0.5787 | region_macro_f1=0.57

,epoch,train_loss,val_loss,val_region_accuracy,val_region_macro_f1,val_region_weighted_f1,val_gender_accuracy,val_gender_macro_f1,val_gender_weighted_f1,joint_score
3,4,0.826126,1.005325,0.627070,0.627974,0.628962,0.944904,0.939544,0.945747,0.783759
4,5,0.711779,1.077650,0.647771,0.647364,0.645409,0.913376,0.906869,0.915351,0.777116
5,6,0.651462,1.016530,0.664013,0.664794,0.664097,0.928344,0.922281,0.929761,0.793538
6,7,0.618750,0.956652,0.670701,0.670595,0.669704,0.945860,0.940580,0.946683,0.805587
7,8,0.599881,0.970952,0.679299,0.680705,0.679995,0.944904,0.939569,0.945757,0.810137


,epoch,train_loss,val_loss,val_region_accuracy,val_region_macro_f1,val_region_weighted_f1,val_gender_accuracy,val_gender_macro_f1,val_gender_weighted_f1,joint_score
3,4,0.826126,1.005325,0.627070,0.627974,0.628962,0.944904,0.939544,0.945747,0.783759
4,5,0.711779,1.077650,0.647771,0.647364,0.645409,0.913376,0.906869,0.915351,0.777116
5,6,0.651462,1.016530,0.664013,0.664794,0.664097,0.928344,0.922281,0.929761,0.793538
6,7,0.618750,0.956652,0.670701,0.670595,0.669704,0.945860,0.940580,0.946683,0.805587
7,8,0.599881,0.970952,0.679299,0.680705,0.679995,0.944904,0.939569,0.945757,0.810137


In [96]:
best_model_path = os.path.join(RUN_DIR, "best_model.pt")
print("Looking for:", best_model_path)

if not os.path.exists(best_model_path):
    raise FileNotFoundError(f"Checkpoint not found: {best_model_path}")

state_dict = torch.load(best_model_path, map_location=DEVICE, weights_only=True)
model.load_state_dict(state_dict)
model.to(DEVICE)
model.eval()

print("Loaded best model:", best_model_path)

Looking for: /content/drive/MyDrive/data/wavlm_runs/wavlm_multitask_region_gender/best_model.pt
Loaded best model: /content/drive/MyDrive/data/wavlm_runs/wavlm_multitask_region_gender/best_model.pt


In [97]:
val_out = run_inference(model, val_loader, DEVICE)
test_out = run_inference(model, test_loader, DEVICE)

print("Validation region metrics:", val_out["region_metrics"])
print("Validation gender metrics:", val_out["gender_metrics"])
print("Test region metrics:", test_out["region_metrics"])
print("Test gender metrics:", test_out["gender_metrics"])

Validation region metrics: {'accuracy': 0.6792993630573249, 'macro_f1': 0.6807048321135719, 'weighted_f1': 0.6799951912835707}
Validation gender metrics: {'accuracy': 0.9449044585987261, 'macro_f1': 0.9395691075569097, 'weighted_f1': 0.9457565135882552}
Test region metrics: {'accuracy': 0.7276422764227642, 'macro_f1': 0.716991125932046, 'weighted_f1': 0.7280279899705109}
Test gender metrics: {'accuracy': 0.9590108401084011, 'macro_f1': 0.9543086372297008, 'weighted_f1': 0.9591846279655865}


In [98]:
region_report = classification_report(
    test_out["region_true"],
    test_out["region_pred"],
    target_names=[REGION_ID2LABEL[i] for i in range(3)],
    digits=4,
    output_dict=True
)

gender_report = classification_report(
    test_out["gender_true"],
    test_out["gender_pred"],
    target_names=[GENDER_ID2LABEL[i] for i in range(2)],
    digits=4,
    output_dict=True
)

print("=== REGION REPORT ===")
print(classification_report(
    test_out["region_true"],
    test_out["region_pred"],
    target_names=[REGION_ID2LABEL[i] for i in range(3)],
    digits=4
))

print("=== GENDER REPORT ===")
print(classification_report(
    test_out["gender_true"],
    test_out["gender_pred"],
    target_names=[GENDER_ID2LABEL[i] for i in range(2)],
    digits=4
))

=== REGION REPORT ===
              precision    recall  f1-score   support

       North     0.7806    0.8155    0.7977      1252
     Central     0.6077    0.6722    0.6383       848
       South     0.7890    0.6538    0.7150       852

    accuracy                         0.7276      2952
   macro avg     0.7257    0.7138    0.7170      2952
weighted avg     0.7333    0.7276    0.7280      2952

=== GENDER REPORT ===
              precision    recall  f1-score   support

        Male     0.9777    0.9603    0.9690      1967
      Female     0.9235    0.9563    0.9397       985

    accuracy                         0.9590      2952
   macro avg     0.9506    0.9583    0.9543      2952
weighted avg     0.9597    0.9590    0.9592      2952



In [99]:
region_cm = confusion_matrix(test_out["region_true"], test_out["region_pred"], labels=[0, 1, 2])
gender_cm = confusion_matrix(test_out["gender_true"], test_out["gender_pred"], labels=[0, 1])

region_cm_path = os.path.join(RUN_DIR, "confusion_matrix_region.png")
gender_cm_path = os.path.join(RUN_DIR, "confusion_matrix_gender.png")

save_confusion_matrix(
    region_cm,
    [REGION_ID2LABEL[i] for i in range(3)],
    "Region Classification Confusion Matrix",
    region_cm_path
)

save_confusion_matrix(
    gender_cm,
    [GENDER_ID2LABEL[i] for i in range(2)],
    "Gender Classification Confusion Matrix",
    gender_cm_path
)

print("Saved:", region_cm_path)
print("Saved:", gender_cm_path)

Saved: /content/drive/MyDrive/data/wavlm_runs/wavlm_multitask_region_gender/confusion_matrix_region.png
Saved: /content/drive/MyDrive/data/wavlm_runs/wavlm_multitask_region_gender/confusion_matrix_gender.png


In [100]:
summary = {
    "model_name": MODEL_NAME,
    "target_sr": TARGET_SR,
    "target_seconds": TARGET_SECONDS,
    "train_size": len(train_df),
    "val_size": len(val_df),
    "test_size": len(test_df),
    "region_metrics_test": test_out["region_metrics"],
    "gender_metrics_test": test_out["gender_metrics"],
    "best_joint_score_val": best_score,
}

save_json(summary, os.path.join(RUN_DIR, "metrics_summary.json"))
save_json(region_report, os.path.join(RUN_DIR, "classification_report_region.json"))
save_json(gender_report, os.path.join(RUN_DIR, "classification_report_gender.json"))

pd.DataFrame(test_out["rows"]).to_csv(os.path.join(RUN_DIR, "test_predictions.csv"), index=False)
save_json(test_out["rows"], os.path.join(RUN_DIR, "test_predictions.json"))

with open(os.path.join(RUN_DIR, "classification_report_region.txt"), "w", encoding="utf-8") as f:
    f.write(classification_report(
        test_out["region_true"],
        test_out["region_pred"],
        target_names=[REGION_ID2LABEL[i] for i in range(3)],
        digits=4
    ))

with open(os.path.join(RUN_DIR, "classification_report_gender.txt"), "w", encoding="utf-8") as f:
    f.write(classification_report(
        test_out["gender_true"],
        test_out["gender_pred"],
        target_names=[GENDER_ID2LABEL[i] for i in range(2)],
        digits=4
    ))

print("Saved all required artifacts to:", RUN_DIR)

Saved all required artifacts to: /content/drive/MyDrive/data/wavlm_runs/wavlm_multitask_region_gender


In [101]:
pd.DataFrame(
    region_cm,
    index=[REGION_ID2LABEL[i] for i in range(3)],
    columns=[REGION_ID2LABEL[i] for i in range(3)]
).to_csv(os.path.join(RUN_DIR, "confusion_matrix_region.csv"))

pd.DataFrame(
    gender_cm,
    index=[GENDER_ID2LABEL[i] for i in range(2)],
    columns=[GENDER_ID2LABEL[i] for i in range(2)]
).to_csv(os.path.join(RUN_DIR, "confusion_matrix_gender.csv"))

print("Saved confusion matrix CSV files.")

Saved confusion matrix CSV files.


In [102]:
torch.save(model.state_dict(), os.path.join(RUN_DIR, "wavlm_multitask_model_state_dict.pt"))

save_json(REGION_ID2LABEL, os.path.join(RUN_DIR, "region_id2label.json"))
save_json(GENDER_ID2LABEL, os.path.join(RUN_DIR, "gender_id2label.json"))

feature_extractor.save_pretrained(os.path.join(RUN_DIR, "feature_extractor"))

print("Saved model state dict and feature extractor.")

Saved model state dict and feature extractor.


In [103]:
print("RUN_DIR:", RUN_DIR)
print("Files in RUN_DIR:")
for p in sorted(os.listdir(RUN_DIR)):
    print(" -", p)

RUN_DIR: /content/drive/MyDrive/data/wavlm_runs/wavlm_multitask_region_gender
Files in RUN_DIR:
 - best_model.pt
 - classification_report_gender.json
 - classification_report_gender.txt
 - classification_report_region.json
 - classification_report_region.txt
 - confusion_matrix_gender.csv
 - confusion_matrix_gender.png
 - confusion_matrix_region.csv
 - confusion_matrix_region.png
 - feature_extractor
 - gender_id2label.json
 - metrics_summary.json
 - region_id2label.json
 - test_predictions.csv
 - test_predictions.json
 - training_history.csv
 - wavlm_multitask_model_state_dict.pt
